In [14]:
# Cell 1: 필수 라이브러리 설치 및 출력 설정
# Playwright가 설치되어 있지 않다면 아래 셀을 먼저 실행하세요.
%pip install playwright
!playwright install chromium

Note: you may need to restart the kernel to use updated packages.


In [38]:
# Cell 2: 라이브러리 import 및 설정
import json
import time
import re
import sys
import os
import subprocess
import tempfile
import inspect

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# 수집할 검색어 개수
MAX_TRENDS = 80

# UI 텍스트 필터링 (제외할 텍스트 목록)
EXCLUDED_TEXTS = {
    "Trends", "트렌드 상태", "트렌드 분석", "검색", "탐색", "실시간 인기", "홈",
    "전 세계", "지금", "에서 무엇을 검색하고 있는지 알아보세요",
    "검색 관심도", "지난 24시간", "이(가) 인기 있는 이유는 무엇일까요?",
    "상세 데이터 검토", "트렌드 데이터팀", "선별한 문제와 이벤트",
    "트렌드 활용법", "언론사", "자선단체", "전 세계에서", "Google 트렌드를 어떻게 사용하고 있는지",
    "확인해보세요", "Google 트렌드란 무엇인가요?", "Google 트렌드의 기본사항",
    "데이터에 관해 알아보기", "로그인", "개인정보처리방침", "고급 Google 트렌드", "도움말", "의견 보내기"
}

# 구글 트렌드 URL
TREND_URL = "https://trends.google.co.kr/trending?geo=KR"

print(f"🔍 구글 트렌드 실시간 인기 검색어 크롤링")
print(f"🔗 URL: {TREND_URL}")
print(f"📦 최대 수집 개수: {MAX_TRENDS}개\n")


🔍 구글 트렌드 실시간 인기 검색어 크롤링
🔗 URL: https://trends.google.co.kr/trending?geo=KR
📦 최대 수집 개수: 80개



In [ ]:
# Cell 3: 크롤링 함수 정의

def crawl_google_trends(headless=True, max_trends=80, excluded_texts=None):
    """
    구글 트렌드 실시간 인기 검색어를 크롤링합니다.
    
    Args:
        headless: 헤드리스 모드 여부
        max_trends: 최대 수집할 검색어 개수
        excluded_texts: 제외할 텍스트 집합 (None이면 EXCLUDED_TEXTS 사용)
        
    Returns:
        dict: {"total_trends": int, "trends": list}
    """
    if excluded_texts is None:
        excluded_texts = EXCLUDED_TEXTS
    
    trends = []
    found_keywords = set()
    
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=headless)
        page = browser.new_page()
        
        # 구글 트렌드 실시간 인기 페이지 접속
        url = "https://trends.google.co.kr/trending?geo=KR"
        print(f"접속 중: {url}")
        
        try:
            # networkidle이 타임아웃될 수 있으므로 load 이벤트로 먼저 시도
            try:
                page.goto(url, wait_until="networkidle", timeout=90000)  # 타임아웃 90초로 증가
            except Exception as e:
                print(f"⚠ networkidle 타임아웃, load 이벤트로 재시도...")
                page.goto(url, wait_until="load", timeout=90000)
            
            time.sleep(8)  # JavaScript 실행 대기
            
            # 트렌드 콘텐츠가 로드될 때까지 대기
            print("트렌드 콘텐츠 로드 대기 중...")
            try:
                # c-wiz 컴포넌트나 트렌드 항목이 나타날 때까지 대기
                page.wait_for_selector("c-wiz, [jsname], [jscontroller]", timeout=20000)
            except:
                pass
            
            # 추가 대기 (동적 콘텐츠 로드)
            time.sleep(5)
            
            # 스크롤하여 더 많은 트렌드 로드
            print("스크롤하여 더 많은 트렌드 로드 중...")
            for scroll_idx in range(20):
                page.evaluate("window.scrollBy(0, window.innerHeight)")
                time.sleep(2)
                # 스크롤 후 추가 대기
                if scroll_idx % 5 == 0:
                    time.sleep(3)
            
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            time.sleep(3)
            
            # 방법 0: 페이지 네비게이션 기반 테이블에서 키워드 추출
            print("\n0. 페이지 네비게이션 테이블에서 키워드 추출 중...")
            page.wait_for_timeout(1000)
            paged_round = 0
            
            # 이전 페이지의 첫 번째 키워드를 저장하여 페이지 변경 확인
            previous_first_keyword = None
            previous_page_keywords = set()  # 이전 페이지의 모든 키워드 저장
            
            while len(trends) < max_trends:
                paged_round += 1
                print(f"  페이지 라운드 {paged_round} 시작...")
                
                if paged_round > 20:
                    print("  최대 페이지 라운드(20) 도달")
                    break
                
                # 테이블이 로드될 때까지 명시적으로 대기
                try:
                    page.wait_for_selector("tbody tr", timeout=10000)
                    page.wait_for_timeout(1000)  # 추가 대기 (동적 콘텐츠 로드)
                except:
                    pass  
                
                # 현재 페이지의 트렌드 수집 (재시도 로직 포함)
                rows = page.query_selector_all("tbody tr")
                
                # 행이 너무 적으면 (1개 이하) 테이블이 완전히 로드되지 않았을 수 있음
                if len(rows) <= 1 and paged_round > 1:
                    print(f"  라운드 {paged_round}: 행이 {len(rows)}개만 발견됨, 테이블 로드 대기 중...")
                    # 추가 대기 및 재시도
                    for retry in range(5):
                        page.wait_for_timeout(1000)
                        rows = page.query_selector_all("tbody tr")
                        if len(rows) > 1:
                            print(f"  라운드 {paged_round}: 재시도 성공 - {len(rows)}개 행 발견")
                            break
                        if retry == 4:
                            print(f"  라운드 {paged_round}: ⚠️ 재시도 실패 - {len(rows)}개 행만 발견")
                
                print(f"  라운드 {paged_round}: {len(rows)}개 행 발견")
                
                if not rows:
                    print(f"  라운드 {paged_round}: 행이 없어 종료")
                    break
                
                # 행이 1개만 있고 이전 라운드에서도 수집이 안 되었다면 종료
                if len(rows) == 1 and paged_round > 1:
                    # 첫 번째 행 확인
                    first_row = rows[0]
                    keyword_elem = first_row.query_selector(".mZ3RIc")
                    if keyword_elem:
                        test_keyword = keyword_elem.inner_text().strip()
                        if test_keyword in found_keywords:
                            print(f"  라운드 {paged_round}: 중복 키워드만 발견, 종료")
                            break
                
                # 현재 페이지의 첫 번째 키워드 확인 (페이지 변경 여부 검증)
                current_first_keyword = None
                current_page_keywords = set()  # 현재 페이지의 모든 키워드
                
                for row in rows:
                    keyword_elem = row.query_selector(".mZ3RIc")
                    if keyword_elem:
                        keyword_text = keyword_elem.inner_text().strip()
                        if keyword_text:
                            current_page_keywords.add(keyword_text)
                            if current_first_keyword is None:
                                current_first_keyword = keyword_text
                
                # 페이지가 변경되지 않았는지 확인 (첫 번째 키워드와 전체 키워드 집합 비교)
                if previous_first_keyword:
                    if current_first_keyword == previous_first_keyword:
                        print(f"  ⚠️ 첫 번째 키워드 동일: {current_first_keyword}")
                        # 전체 키워드 집합도 비교
                        if current_page_keywords == previous_page_keywords:
                            print(f"  ⚠️ 페이지 내용이 완전히 동일함 (중복 키워드: {len(current_page_keywords)}개), 종료")
                            break
                        else:
                            # 첫 번째 키워드는 같지만 다른 키워드가 있음 - 페이지가 변경되었을 수 있음
                            new_keywords = current_page_keywords - previous_page_keywords
                            if len(new_keywords) > 0:
                                print(f"  ✅ 첫 번째 키워드는 같지만 새로운 키워드 {len(new_keywords)}개 발견, 계속 진행")
                            else:
                                print(f"  ⚠️ 첫 번째 키워드는 같고 새로운 키워드 없음, 종료")
                                break
                
                previous_first_keyword = current_first_keyword
                previous_page_keywords = current_page_keywords.copy()
                
                new_items = 0
                skipped_duplicates = 0
                for row in rows:
                    keyword_elem = row.query_selector(".mZ3RIc")
                    if not keyword_elem:
                        continue
                    text = keyword_elem.inner_text().strip()
                    if text and 1 < len(text) < 100:
                        if (text not in excluded_texts and
                            text not in found_keywords and
                            not any(excluded in text for excluded in excluded_texts)):
                            link_url = ""
                            link_elem = row.query_selector("a")
                            if link_elem:
                                href = link_elem.get_attribute("href") or ""
                                if href.startswith("http"):
                                    link_url = href
                                elif href.startswith("/"):
                                    link_url = f"https://trends.google.co.kr{href}"
                                elif href:
                                    link_url = f"https://trends.google.co.kr/{href}"
                            found_keywords.add(text)
                            trends.append({
                                "keyword": text,
                                "link": link_url
                            })
                            new_items += 1
                            if len(trends) >= max_trends:
                                break
                        else:
                            if text in found_keywords:
                                skipped_duplicates += 1
                
                print(f"  라운드 {paged_round}: {new_items}개 수집 (총 {len(trends)}개)")
                
                if len(trends) >= max_trends:
                    print(f"  목표 개수({max_trends}) 도달")
                    break
                
                if new_items == 0:
                    print(f"  라운드 {paged_round}: 새 항목 없음 (수집: {len(trends)}/{max_trends}개)")
                    break
                
                # 다음 페이지 버튼 찾기
                # DOM 구조: 
                # div.enOdEe-wZVHId-gruSEe > 
                #   div.enOdEe-wZVHId-gruSEe-UbuQg > 
                #     div.enOdEe-wZVHId-gruSEe-yXBf7b > 
                #       span > 
                #         button[jsname='ViaHrd'][aria-label='다음 페이지로 이동']
                #         class="pYTkkf-Bz112c-LgbsSe pYTkkf-Bz112c-LgbsSe-OWXEXe-SfQLQb-suEOdc enOdEe-wZVHId-gruSEe-LgbsSe"
                next_button = None
                next_button_selectors = [
                    # 방법 1: jsname 기반 (가장 정확)
                    "button[jsname='ViaHrd']",
                    "[jsname='ViaHrd']",
                    
                    # 방법 2: 부모 구조를 통한 정확한 선택
                    ".enOdEe-wZVHId-gruSEe .enOdEe-wZVHId-gruSEe-UbuQg .enOdEe-wZVHId-gruSEe-yXBf7b button[jsname='ViaHrd']",
                    ".enOdEe-wZVHId-gruSEe .enOdEe-wZVHId-gruSEe-UbuQg .enOdEe-wZVHId-gruSEe-yXBf7b span button[jsname='ViaHrd']",
                    ".enOdEe-wZVHId-gruSEe .enOdEe-wZVHId-gruSEe-UbuQg .enOdEe-wZVHId-gruSEe-yXBf7b button[aria-label='다음 페이지로 이동']",
                    
                    # 방법 3: 클래스 조합 + aria-label
                    "button.enOdEe-wZVHId-gruSEe-LgbsSe[aria-label='다음 페이지로 이동']",
                    "button[class*='enOdEe-wZVHId-gruSEe-LgbsSe'][aria-label='다음 페이지로 이동']",
                    "button[class*='pYTkkf-Bz112c-LgbsSe'][aria-label='다음 페이지로 이동']",
                    
                    # 방법 4: jscontroller + aria-label 조합
                    "button[jscontroller='PIVayb'][aria-label='다음 페이지로 이동']",
                    "[jscontroller='PIVayb'][aria-label='다음 페이지로 이동']",
                    
                    # 방법 5: aria-label만 (백업)
                    "button[aria-label='다음 페이지로 이동']",
                    "[aria-label='다음 페이지로 이동']",
                    
                    # 방법 6: 부모 구조만으로 찾기 (클래스 기반)
                    ".enOdEe-wZVHId-gruSEe-yXBf7b button[jsname='ViaHrd']",
                    ".enOdEe-wZVHId-gruSEe-yXBf7b button[aria-label='다음 페이지로 이동']",
                    ".enOdEe-wZVHId-gruSEe-yXBf7b button",
                ]
                
                # 버튼 찾기 전에 페이지가 완전히 로드될 때까지 대기
                page.wait_for_timeout(1000)
                
                for selector in next_button_selectors:
                    try:
                        next_button = page.query_selector(selector)
                        if next_button:
                            # 버튼이 보이면 비활성화 상태와 관계없이 사용 (JavaScript로 강제 클릭 가능)
                            is_visible = next_button.is_visible()
                            
                            if is_visible:
                                # 비활성화 상태 확인 (로그용)
                                is_enabled = next_button.is_enabled()
                                is_disabled = False
                                try:
                                    disabled_attr = next_button.get_attribute("disabled")
                                    aria_disabled = next_button.get_attribute("aria-disabled")
                                    if disabled_attr is not None or aria_disabled == "true":
                                        is_disabled = True
                                except:
                                    pass
                                
                                if is_disabled or not is_enabled:
                                    print(f"  다음 페이지 버튼 발견 (비활성화 상태이지만 강제 클릭 시도): {selector}")
                                else:
                                    print(f"  다음 페이지 버튼 발견: {selector}")
                                break
                            else:
                                next_button = None
                    except Exception as e:
                        next_button = None
                        continue
                
                # 버튼을 찾지 못한 경우, 페이지 하단으로 스크롤하여 버튼이 로드되도록 시도
                if not next_button:
                    print("  버튼을 찾지 못함, 페이지 하단으로 스크롤하여 재시도...")
                    page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    page.wait_for_timeout(1500)
                    
                    # 다시 버튼 찾기 시도
                    for selector in next_button_selectors[:5]:  # 상위 5개 셀렉터만 재시도
                        try:
                            next_button = page.query_selector(selector)
                            if next_button:
                                is_visible = next_button.is_visible()
                                
                                # 버튼이 보이면 비활성화 상태와 관계없이 사용
                                if is_visible:
                                    is_enabled = next_button.is_enabled()
                                    is_disabled = False
                                    try:
                                        disabled_attr = next_button.get_attribute("disabled")
                                        aria_disabled = next_button.get_attribute("aria-disabled")
                                        if disabled_attr is not None or aria_disabled == "true":
                                            is_disabled = True
                                    except:
                                        pass
                                    
                                    if is_disabled or not is_enabled:
                                        print(f"  스크롤 후 버튼 발견 (비활성화 상태이지만 강제 클릭 시도): {selector}")
                                    else:
                                        print(f"  스크롤 후 버튼 발견: {selector}")
                                    break
                                else:
                                    next_button = None
                        except:
                            next_button = None
                            continue
                
                if not next_button:
                    print(f"  라운드 {paged_round}: 다음 페이지 버튼 없음 (수집: {len(trends)}/{max_trends}개)")
                    break
                
                # 다음 페이지로 이동 (모든 페이지에서 동일한 로직 사용)
                try:
                    # 버튼이 뷰포트에 보이도록 스크롤
                    next_button.scroll_into_view_if_needed()
                    page.wait_for_timeout(300)
                    
                    # 이전 페이지의 첫 번째 키워드 저장 (클릭 전)
                    old_first_keyword = previous_first_keyword
                    
                    # JavaScript로 직접 클릭 (강화된 버전)
                    click_success = False
                    try:
                        # JavaScript로 클릭 이벤트 직접 트리거 (비활성화 상태 무시하고 강제 클릭)
                        next_button.evaluate("""
                            el => {
                                // 비활성화 속성 제거 (일시적으로)
                                const wasDisabled = el.hasAttribute('disabled');
                                const wasAriaDisabled = el.getAttribute('aria-disabled');
                                if (wasDisabled) {
                                    el.removeAttribute('disabled');
                                }
                                if (wasAriaDisabled === 'true') {
                                    el.setAttribute('aria-disabled', 'false');
                                }
                                
                                // 1. onclick 핸들러 직접 호출
                                if (el.onclick) {
                                    try {
                                        el.onclick();
                                    } catch(e) {}
                                }
                                
                                // 2. click() 메서드 호출
                                try {
                                    el.click();
                                } catch(e) {}
                                
                                // 3. MouseEvent로 클릭 이벤트 발생
                                const clickEvent = new MouseEvent('click', {
                                    bubbles: true,
                                    cancelable: true,
                                    view: window,
                                    detail: 1,
                                    buttons: 1
                                });
                                el.dispatchEvent(clickEvent);
                                
                                // 4. mousedown, mouseup 이벤트도 발생
                                const mouseDownEvent = new MouseEvent('mousedown', {
                                    bubbles: true,
                                    cancelable: true,
                                    view: window,
                                    detail: 1,
                                    buttons: 1
                                });
                                el.dispatchEvent(mouseDownEvent);
                                
                                const mouseUpEvent = new MouseEvent('mouseup', {
                                    bubbles: true,
                                    cancelable: true,
                                    view: window,
                                    detail: 1,
                                    buttons: 1
                                });
                                el.dispatchEvent(mouseUpEvent);
                                
                                // 5. jsaction이 있으면 직접 실행
                                const jsaction = el.getAttribute('jsaction');
                                if (jsaction) {
                                    // click:h5M12e 같은 패턴 처리
                                    const actions = jsaction.split(';');
                                    for (let action of actions) {
                                        if (action.includes('click:')) {
                                            const handlerName = action.split(':')[1];
                                            if (window[handlerName]) {
                                                try {
                                                    window[handlerName](new Event('click'));
                                                } catch(e) {}
                                            }
                                        }
                                    }
                                }
                                
                                // 6. jsname 속성을 이용한 직접 호출
                                const jsname = el.getAttribute('jsname');
                                if (jsname === 'ViaHrd') {
                                    // Google의 내부 핸들러 직접 호출 시도
                                    try {
                                        const clickHandler = el.onclick || el.getAttribute('onclick');
                                        if (clickHandler) {
                                            eval(clickHandler.toString());
                                        }
                                    } catch(e) {}
                                }
                            }
                        """)
                        click_success = True
                    except Exception as js_error:
                        pass  
                    
                    # Playwright 클릭 (JavaScript가 실패한 경우)
                    if not click_success:
                        try:
                            next_button.click(timeout=5000)
                            click_success = True
                        except:
                            try:
                                next_button.click(force=True, timeout=5000)
                                click_success = True
                            except:
                                pass
                    
                    if not click_success:
                        # aria-label로 버튼 찾아서 JavaScript로 클릭
                        try:
                            page.evaluate("""
                                () => {
                                    const btn = document.querySelector("button[jsname='ViaHrd']");
                                    if (btn) {
                                        btn.click();
                                        const clickEvent = new MouseEvent('click', {
                                            bubbles: true,
                                            cancelable: true,
                                            view: window
                                        });
                                        btn.dispatchEvent(clickEvent);
                                        return true;
                                    }
                                    return false;
                                }
                            """)
                            click_success = True
                        except:
                            pass
                    
                    if not click_success:
                        raise Exception("모든 클릭 방법 실패")
                    
                    # 페이지 로드 대기 (강화된 버전)
                    page.wait_for_timeout(2000)  # 클릭 후 초기 대기
                    
                    # 페이지가 로드될 때까지 대기
                    try:
                        page.wait_for_load_state("networkidle", timeout=15000)
                    except:
                        pass  # 로그 제거
                    
                    # 스크롤하여 테이블이 완전히 로드되도록 함
                    page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    page.wait_for_timeout(1000)
                    page.evaluate("window.scrollTo(0, 0)")
                    page.wait_for_timeout(1000)
                    
                    # 테이블이 업데이트될 때까지 명시적으로 대기 (강화된 버전)
                    try:
                        # 테이블이 업데이트될 때까지 대기 (최대 15초)
                        table_updated = False
                        min_rows_expected = 20  # 최소 예상 행 수
                        last_row_count = 0
                        stable_count = 0
                        
                        for wait_attempt in range(15):
                            page.wait_for_timeout(500)  # 0.5초마다 확인
                            
                            # 테이블 행 확인
                            new_rows = page.query_selector_all("tbody tr")
                            current_row_count = len(new_rows)
                            
                            # 행 수가 안정적으로 유지되는지 확인
                            if current_row_count == last_row_count and current_row_count > 0:
                                stable_count += 1
                            else:
                                stable_count = 0
                            last_row_count = current_row_count
                            
                            # 행 수가 충분하고 안정적이며, 첫 번째 키워드가 변경되었는지 확인
                            if current_row_count >= min_rows_expected and stable_count >= 2:
                                # 첫 번째 행의 키워드 확인
                                first_row = new_rows[0]
                                keyword_elem = first_row.query_selector(".mZ3RIc")
                                if keyword_elem:
                                    new_first_keyword = keyword_elem.inner_text().strip()
                                    
                                    # 이전 페이지의 첫 번째 키워드와 다르면 페이지가 변경된 것
                                    if old_first_keyword is None or new_first_keyword != old_first_keyword:
                                        # 추가 검증: 마지막 행의 키워드도 확인
                                        last_row = new_rows[-1]
                                        last_keyword_elem = last_row.query_selector(".mZ3RIc")
                                        if last_keyword_elem:
                                            last_keyword = last_keyword_elem.inner_text().strip()
                                            # 첫 번째와 마지막 키워드가 모두 다르면 확실히 페이지가 변경된 것
                                            if last_keyword not in found_keywords:
                                                table_updated = True
                                                break
                            
                            if wait_attempt == 14:
                                # 타임아웃이어도 첫 번째 키워드가 변경되었는지 확인
                                if current_row_count > 0:
                                    first_row = new_rows[0]
                                    keyword_elem = first_row.query_selector(".mZ3RIc")
                                    if keyword_elem:
                                        final_keyword = keyword_elem.inner_text().strip()
                                        if final_keyword != old_first_keyword:
                                            table_updated = True
                    except Exception as e:
                        pass  # 로그 제거
                    
                    # 추가 대기 및 스크롤 (동적 콘텐츠 로드)
                    page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    page.wait_for_timeout(1500)  # 대기 시간
                    page.evaluate("window.scrollTo(0, 0)")
                    page.wait_for_timeout(500)
                except Exception as e:
                    print(f"  다음 페이지 이동 실패: {e}")
                    break
            
            print(f"\n방법 0 완료: 총 {len(trends)}개 트렌드 수집")
            
            # 방법 1, 2, 3 제거됨 (방법 0만 사용)
            
            # 최종 결과 정리
            trends = trends[:max_trends]
            
            print(f"\n✅ 총 {len(trends)}개 트렌드 키워드 수집 완료")
            
        except Exception as e:
            print(f"⚠ 크롤링 오류: {e}")
            import traceback
            traceback.print_exc()
        
        finally:
            browser.close()
    
    # 결과 반환
    return {
        "total_trends": len(trends),
        "trends": trends
    }


In [42]:
# Cell 4: 실행 및 결과 저장

print("▶ 구글 트렌드 실시간 인기 검색어 크롤링을 시작합니다...")
print(f"📦 최대 수집 개수: {MAX_TRENDS}개\n")

# 함수의 소스 코드를 가져와서 Python 파일로 저장
# 이렇게 하면 실제 Python 코드로 작성된 함수를 별도 프로세스에서 실행 가능
function_source = inspect.getsource(crawl_google_trends)

# 실행 스크립트 생성 (함수 + 실행 코드)
script_content = f"""from playwright.sync_api import sync_playwright
import json
import time
import re

# 함수 정의
{function_source}

# 설정값
EXCLUDED_TEXTS = {EXCLUDED_TEXTS!r}

# 함수 실행
if __name__ == "__main__":
    result = crawl_google_trends(
        headless={NOTEBOOK_HEADLESS!r},
        max_trends={MAX_TRENDS},
        excluded_texts=EXCLUDED_TEXTS
    )
    
    # JSON 파일로 저장
    output_json = "google_trends_results.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    
    # 결과 미리보기
    print(f"\\n📋 수집된 트렌드 키워드 미리보기 (처음 15개):")
    for idx, trend in enumerate(result["trends"][:15], 1):
        print(f"  {{idx}}. {{trend['keyword']}}")
        if trend['link']:
            print(f"     링크: {{trend['link'][:70]}}...")
"""

# 임시 파일에 스크립트 저장
fd, script_path = tempfile.mkstemp(suffix="_crawl_google_trends.py", text=True)
os.close(fd)
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)

try:
    # 별도 프로세스에서 실행 (Windows 인코딩 문제 해결)
    completed = subprocess.run(
        [sys.executable, script_path],
        capture_output=True,
        text=True,
        encoding='utf-8',  # UTF-8 인코딩 명시
        errors='replace',  # 인코딩 에러 시 대체 문자 사용
        check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
    else:
        # JSON 파일 읽기
        output_json = "google_trends_results.json"
        if os.path.exists(output_json):
            with open(output_json, "r", encoding="utf-8") as f:
                result = json.load(f)
            print(f"\n✅ 크롤링 완료: 총 {result['total_trends']}개 키워드 수집")
finally:
    # 임시 파일 삭제
    try:
        os.remove(script_path)
    except Exception:
        pass


▶ 구글 트렌드 실시간 인기 검색어 크롤링을 시작합니다...
📦 최대 수집 개수: 80개



27594

접속 중: https://trends.google.co.kr/trending?geo=KR
트렌드 콘텐츠 로드 대기 중...
스크롤하여 더 많은 트렌드 로드 중...

0. 페이지 네비게이션 테이블에서 키워드 추출 중...
  페이지 라운드 1 시작...
  라운드 1: 26개 행 발견
  라운드 1: 25개 수집 (총 25개)
  다음 페이지 버튼 발견: button[jsname='ViaHrd']
  페이지 라운드 2 시작...
  라운드 2: 26개 행 발견
  라운드 2: 24개 수집 (총 49개)
  다음 페이지 버튼 발견: button[jsname='ViaHrd']
  페이지 라운드 3 시작...
  라운드 3: 행이 1개만 발견됨, 테이블 로드 대기 중...
  라운드 3: ⚠️ 재시도 실패 - 1개 행만 발견
  라운드 3: 1개 행 발견
  라운드 3: 0개 수집 (총 49개)
  라운드 3: 새 항목 없음 (수집: 49/80개)

방법 0 완료: 총 49개 트렌드 수집

✅ 총 49개 트렌드 키워드 수집 완료
✅ JSON 파일 저장 완료: google_trends_results.json

📋 수집된 트렌드 키워드 미리보기 (처음 15개):
  1. 토트넘 대 슬라비아 프라하
     링크: https://trends.google.co.kr/trends/explore?q=%EC%86%90%ED%9D%A5%EB%AF%...
  2. 바이에른 대 스포르팅
     링크: https://trends.google.co.kr/trends/explore?q=%EC%B1%94%ED%94%BC%EC%96%...
  3. 인테르 대 리버풀
     링크: https://trends.google.co.kr/trends/explore?q=%EC%9D%B8%ED%85%8C%EB%A5%...
  4. 김지미
  5. 인요한
     링크: https://trends.google.co.kr/trends/explore?q=%EC%9D%B4%EC%86%8C%ED%9D%...
